In [5]:
import os

BASE_PATH = "/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets"  
# adjust if your exact folder name differs — the one you gave was:
# /kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets

# ---- Step 2: List files ----
for root, dirs, files in os.walk(BASE_PATH):
    for f in files:
        print(os.path.join(root, f))

# ---- Step 3: CoNLL parser ----
def parse_conll(file_path):
    sentences, tags = [], []
    tokens, tag_seq = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    sentences.append(tokens)
                    tags.append(tag_seq)
                    tokens, tag_seq = [], []
            else:
                parts = line.split("\t")
                if len(parts) < 2:
                    continue
                token, tag = parts[0], parts[-1]
                tokens.append(token)
                tag_seq.append(tag)
    if tokens:
        sentences.append(tokens)
        tags.append(tag_seq)
    return sentences, tags

hindi_train_path = os.path.join(BASE_PATH, "Hindi", "Hindi-train.txt")
hindi_dev_path   = os.path.join(BASE_PATH, "Hindi", "Hindi-dev.txt")
hindi_test_path  = os.path.join(BASE_PATH, "Hindi", "Hindi-test.txt")

train_sents, train_tags = parse_conll(hindi_train_path)
dev_sents, dev_tags     = parse_conll(hindi_dev_path)
test_sents, test_tags   = parse_conll(hindi_test_path)

print(f"Train sentences: {len(train_sents)}  (expected 11076)")
print(f"Dev sentences:   {len(dev_sents)}  (expected 1389)")
print(f"Test sentences:  {len(test_sents)}  (expected 1389)")

# ---- Step 4: Label distribution + error check ----
from collections import Counter

all_tags = [t for seq in train_tags for t in seq]
tag_counts = Counter(all_tags)
print("\nLabel distribution (train):")
for tag, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {tag}: {count}")

# Check for known error patterns
valid_prefixes = ("B-", "I-", "O")
valid_types = {"NEP", "NEL", "NEO", "NEAR", "NEN", "NETI"}

bad_tags = set()
for tag in tag_counts:
    if tag == "O":
        continue
    if not tag.startswith(("B-", "I-")):
        bad_tags.add(tag)
    else:
        ttype = tag.split("-", 1)[1]
        if ttype not in valid_types:
            bad_tags.add(tag)

print(f"\nBad/unexpected tags found: {bad_tags if bad_tags else 'None — data is clean'}")

# Sample check
print("\nFirst training example:")
for tok, tag in zip(train_sents[0], train_tags[0]):
    print(f"  {tok}\t{tag}")

/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Hindi/Hindi-dev.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Hindi/Hindi-train.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Hindi/Hindi-test.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Odia/Odia-test.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Odia/Odia-dev.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Odia/Odia-train.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Telugu/Telugu-train.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Telugu/Telugu-dev.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Telugu/Telugu-test.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-f

In [6]:
!pip install -q peft accelerate

In [7]:
!pip uninstall -y torchao
!pip install -q --upgrade peft accelerate transformers

In [3]:
!pip install -q --use-pep517 pytorch-crf seqeval transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
lengths = [
    len(tokenizer(words, is_split_into_words=True)["input_ids"])
    for words in train_sents
]

print("Maximum subword length:", max(lengths))
print("Sentences > 128:", sum(x > 128 for x in lengths))
print("Sentences > 256:", sum(x > 256 for x in lengths))

Maximum subword length: 215
Sentences > 128: 19
Sentences > 256: 0


In [9]:
import json
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

unique_tags = sorted(set(tag for sentence_tags in train_tags for tag in sentence_tags))
tag2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2tag = {idx: tag for tag, idx in tag2id.items()}

NUM_TAGS = len(tag2id)
O_TAG_INDEX = tag2id["O"]

print("Device:", DEVICE)
print("Number of BIO labels:", NUM_TAGS)
print("O tag index:", O_TAG_INDEX)
print("tag2id:", tag2id)

with open("/kaggle/working/tag2id.json", "w", encoding="utf-8") as f:
    json.dump(tag2id, f, ensure_ascii=False, indent=2)

print("Saved: /kaggle/working/tag2id.json")

Device: cuda
Number of BIO labels: 13
O tag index: 12
tag2id: {'B-NEAR': 0, 'B-NEL': 1, 'B-NEN': 2, 'B-NEO': 3, 'B-NEP': 4, 'B-NETI': 5, 'I-NEAR': 6, 'I-NEL': 7, 'I-NEN': 8, 'I-NEO': 9, 'I-NEP': 10, 'I-NETI': 11, 'O': 12}
Saved: /kaggle/working/tag2id.json


In [10]:
import json
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open("/kaggle/working/tag2id.json", "r", encoding="utf-8") as f:
    tag2id = json.load(f)

id2tag = {idx: tag for tag, idx in tag2id.items()}
NUM_TAGS = len(tag2id)
O_TAG_INDEX = tag2id["O"]

print("Device:", DEVICE)
print("Number of BIO labels:", NUM_TAGS)
print("O tag index:", O_TAG_INDEX)
print("Tag mapping:", tag2id)

Device: cuda
Number of BIO labels: 13
O tag index: 12
Tag mapping: {'B-NEAR': 0, 'B-NEL': 1, 'B-NEN': 2, 'B-NEO': 3, 'B-NEP': 4, 'B-NETI': 5, 'I-NEAR': 6, 'I-NEL': 7, 'I-NEN': 8, 'I-NEO': 9, 'I-NEP': 10, 'I-NETI': 11, 'O': 12}


In [11]:
import os
import json
import time
import copy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score

In [12]:
class NERDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, max_length=128):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        word_tags = self.tags[idx]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        word_ids = encoding.word_ids(batch_index=0)

        # Must be a valid label ID because pytorch-crf indexes into it.
        label_ids = torch.full_like(
            input_ids,
            fill_value=O_TAG_INDEX,
            dtype=torch.long
        )

        # CRF sees only first subword of each original word.
        crf_mask = torch.zeros_like(input_ids, dtype=torch.bool)

        previous_word_id = None
        for token_index, word_id in enumerate(word_ids):
            if word_id is None:
                continue  # ignores <s>, </s>, padding

            if word_id != previous_word_id:
                label_ids[token_index] = tag2id[word_tags[word_id]]
                crf_mask[token_index] = True

            previous_word_id = word_id

        # pytorch-crf requires the first sequence position to be valid.
        # XLM-R has <s> at position 0, so we include it with a dummy O label.
        # It will be excluded from seqeval evaluation below.
        crf_mask[0] = True
        label_ids[0] = O_TAG_INDEX

        return input_ids, attention_mask, label_ids, crf_mask


def collate_fn(batch):
    (
        input_ids,
        attention_masks,
        label_ids,
        crf_masks,
        first_subword_masks,
    ) = zip(*batch)

    return (
        torch.stack(input_ids),
        torch.stack(attention_masks),
        torch.stack(label_ids),
        torch.stack(crf_masks),
        torch.stack(first_subword_masks),
    )

In [13]:
!pip install -q peft accelerate

from peft import LoraConfig, TaskType, get_peft_model

print("PEFT / LoRA imports loaded successfully")

PEFT / LoRA imports loaded successfully


In [14]:
class XLMR_LoRA_CRF(nn.Module):
    def __init__(
        self,
        model_name="xlm-roberta-base",
        num_tags=NUM_TAGS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.10,
        dropout=0.30,
    ):
        super().__init__()

        base_model = AutoModel.from_pretrained(model_name)

        lora_config = LoraConfig(
            task_type=TaskType.TOKEN_CLS,
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["query", "value"],
            bias="none",
        )

        self.transformer = get_peft_model(base_model, lora_config)
        hidden_size = self.transformer.config.hidden_size

        # This head remains trainable because it is newly initialized.
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, mask=None):
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(hidden_states)

        if labels is not None:
            return -self.crf(
                emissions,
                labels,
                mask=mask,
                reduction="mean",
            )

        return self.crf.decode(emissions, mask=mask)

    def parameter_counts(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(
            p.numel() for p in self.parameters()
            if p.requires_grad
        )
        return total, trainable

In [19]:
def train_one_epoch_lora(model, loader, optimizer, scheduler):
    model.train()
    running_loss = 0.0
    for (
        input_ids,
        attention_mask,
        labels,
        crf_mask,
        first_subword_mask,
    ) in loader:
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        attention_mask = attention_mask.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        crf_mask = crf_mask.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            mask=crf_mask,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [20]:
def run_lora_finetuning(
    model_name="xlm-roberta-base",
    max_length=128,
    batch_size=8,
    learning_rate=2e-4,
    epochs=10,
    patience=3,
    r=8,
    lora_alpha=16,
    lora_dropout=0.10,
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="right",
    )

    train_ds = NERDataset(train_sents, train_tags, tokenizer, max_length)
    dev_ds = NERDataset(dev_sents, dev_tags, tokenizer, max_length)
    test_ds = NERDataset(test_sents, test_tags, tokenizer, max_length)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True,
    )

    model = XLMR_LoRA_CRF(
        model_name=model_name,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
    ).to(DEVICE)

    total_params, trainable_params = model.parameter_counts()

    print("\n========== Tier 3: LoRA Fine-Tuning ==========")
    print(f"Backbone: {model_name}")
    print(f"LoRA rank: {r}, alpha: {lora_alpha}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=0.01,
    )

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.10 * total_steps),
        num_training_steps=total_steps,
    )

    best_dev_f1 = -1.0
    best_epoch = 0
    patience_counter = 0
    start_time = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        train_loss = train_one_epoch_lora(
            model,
            train_loader,
            optimizer,
            scheduler,
        )

        dev_f1, _, _, _ = evaluate(model, dev_loader, id2tag)
        epoch_seconds = time.time() - epoch_start

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"dev_f1={dev_f1:.4f} | "
            f"time={epoch_seconds:.1f}s"
        )

        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_epoch = epoch
            patience_counter = 0

            checkpoint = {
                "epoch": epoch,
                "best_dev_f1": best_dev_f1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "tag2id": tag2id,
                "lora_config": {
                    "r": r,
                    "lora_alpha": lora_alpha,
                    "lora_dropout": lora_dropout,
                    "target_modules": ["query", "value"],
                },
            }

            torch.save(
                checkpoint,
                "/kaggle/working/xlmr_lora_best_checkpoint.pt",
            )
            print("  Saved best checkpoint.")

        else:
            patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    checkpoint = torch.load(
        "/kaggle/working/xlmr_lora_best_checkpoint.pt",
        map_location=DEVICE,
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    test_start = time.time()
    test_f1, test_report, _, _ = evaluate(model, test_loader, id2tag)
    test_seconds = time.time() - test_start

    elapsed_seconds = time.time() - start_time
    peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available()
        else 0
    )

    # Save only LoRA adapter weights separately.
    model.transformer.save_pretrained(
        "/kaggle/working/xlmr_hindi_ner_lora_adapter"
    )

    # Save CRF + classifier head, which LoRA adapter files do not include.
    torch.save(
        {
            "classifier_state_dict": model.classifier.state_dict(),
            "crf_state_dict": model.crf.state_dict(),
            "tag2id": tag2id,
        },
        "/kaggle/working/xlmr_lora_ner_head.pt",
    )

    results = {
        "tier": "tier3_lora",
        "backbone": model_name,
        "best_epoch": best_epoch,
        "best_dev_f1": round(best_dev_f1, 4),
        "test_f1": round(test_f1, 4),
        "train_time_seconds": round(elapsed_seconds, 2),
        "test_inference_seconds": round(test_seconds, 3),
        "peak_gpu_memory_mb": round(peak_gpu_memory_mb, 2),
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "trainable_parameter_pct": round(
            100 * trainable_params / total_params,
            4,
        ),
        "lora_rank": r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "test_report": test_report,
    }

    with open(
        "/kaggle/working/tier3_lora_results.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("\n========== Tier 3 Final Results ==========")
    print(f"Best epoch: {best_epoch}")
    print(f"Best dev F1: {best_dev_f1:.4f}")
    print(f"Test F1: {test_f1:.4f}")
    print(f"Total training time: {elapsed_seconds / 60:.2f} min")
    print(f"Peak GPU memory: {peak_gpu_memory_mb:.2f} MB")
    print("\n" + test_report)

    return model, results

In [17]:
from seqeval.metrics import classification_report, f1_score
import torch

@torch.no_grad()
def evaluate(model, loader, id2tag):
    model.eval()

    all_predictions = []
    all_gold_labels = []

    for (
        input_ids,
        attention_mask,
        labels,
        crf_mask,
        first_subword_mask,
    ) in loader:

        input_ids = input_ids.to(DEVICE, non_blocking=True)
        attention_mask = attention_mask.to(DEVICE, non_blocking=True)
        crf_mask = crf_mask.to(DEVICE, non_blocking=True)

        decoded_sequences = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask=crf_mask,
        )

        for batch_index, decoded_ids in enumerate(decoded_sequences):
            active_positions = torch.where(
                crf_mask[batch_index]
            )[0].cpu().tolist()

            position_to_decoded_index = {
                original_position: decoded_index
                for decoded_index, original_position
                in enumerate(active_positions)
            }

            word_positions = torch.where(
                first_subword_mask[batch_index]
            )[0].cpu().tolist()

            true_ids = labels[
                batch_index,
                word_positions
            ].cpu().tolist()

            predicted_ids = [
                decoded_ids[position_to_decoded_index[position]]
                for position in word_positions
            ]

            assert len(true_ids) == len(predicted_ids), (
                f"Gold/prediction mismatch: "
                f"{len(true_ids)} versus {len(predicted_ids)}"
            )

            all_gold_labels.append(
                [id2tag[label_id] for label_id in true_ids]
            )
            all_predictions.append(
                [id2tag[label_id] for label_id in predicted_ids]
            )

    report = classification_report(
        all_gold_labels,
        all_predictions,
        digits=4,
        zero_division=0,
    )

    micro_f1 = f1_score(
        all_gold_labels,
        all_predictions,
        zero_division=0,
    )

    return micro_f1, report, all_predictions, all_gold_labels

In [22]:
class NERDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, max_length=128):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        word_tags = self.tags[idx]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        word_ids = encoding.word_ids(batch_index=0)

        # All active CRF positions must have a valid label ID.
        # O is a safe dummy tag for special and padding positions.
        label_ids = torch.full_like(
            input_ids,
            fill_value=O_TAG_INDEX,
            dtype=torch.long,
        )

        # Save the position of the first subword for each original word.
        # Used only for evaluation, not as the CRF mask.
        first_subword_mask = torch.zeros_like(
            input_ids,
            dtype=torch.bool,
        )

        previous_word_id = None

        for token_index, word_id in enumerate(word_ids):
            if word_id is None:
                continue

            original_tag = word_tags[word_id]

            if word_id != previous_word_id:
                # First subword retains original BIO label.
                label_ids[token_index] = tag2id[original_tag]
                first_subword_mask[token_index] = True

            else:
                # Continuation subword must preserve entity continuation.
                # B-NEP -> I-NEP, I-NEP -> I-NEP, O -> O.
                if original_tag.startswith("B-"):
                    continuation_tag = "I-" + original_tag[2:]
                else:
                    continuation_tag = original_tag

                label_ids[token_index] = tag2id[continuation_tag]

            previous_word_id = word_id

        # CRF requires a contiguous valid mask: every non-pad position,
        # including <s> and </s>, is active.
        crf_mask = attention_mask.bool()

        return (
            input_ids,
            attention_mask,
            label_ids,
            crf_mask,
            first_subword_mask,
        )

@torch.no_grad()
def evaluate(model, loader, id2tag):
    model.eval()

    all_predictions = []
    all_gold_labels = []

    for (
        input_ids,
        attention_mask,
        labels,
        crf_mask,
        first_subword_mask,
    ) in loader:

        input_ids = input_ids.to(DEVICE, non_blocking=True)
        attention_mask = attention_mask.to(DEVICE, non_blocking=True)
        crf_mask = crf_mask.to(DEVICE, non_blocking=True)

        decoded_sequences = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask=crf_mask,
        )

        for batch_index, decoded_ids in enumerate(decoded_sequences):
            active_positions = torch.where(
                crf_mask[batch_index]
            )[0].cpu().tolist()

            position_to_decoded_index = {
                original_position: decoded_index
                for decoded_index, original_position
                in enumerate(active_positions)
            }

            word_positions = torch.where(
                first_subword_mask[batch_index]
            )[0].cpu().tolist()

            true_ids = labels[
                batch_index,
                word_positions
            ].cpu().tolist()

            predicted_ids = [
                decoded_ids[position_to_decoded_index[position]]
                for position in word_positions
            ]

            assert len(true_ids) == len(predicted_ids), (
                f"Gold/prediction mismatch: "
                f"{len(true_ids)} versus {len(predicted_ids)}"
            )

            all_gold_labels.append(
                [id2tag[label_id] for label_id in true_ids]
            )
            all_predictions.append(
                [id2tag[label_id] for label_id in predicted_ids]
            )

    report = classification_report(
        all_gold_labels,
        all_predictions,
        digits=4,
        zero_division=0,
    )

    micro_f1 = f1_score(
        all_gold_labels,
        all_predictions,
        zero_division=0,
    )

    return micro_f1, report, all_predictions, all_gold_labels

In [ ]:
lora_model, tier3_results = run_lora_finetuning(
    max_length=128,
    batch_size=8,
    learning_rate=1e-4,
    epochs=10,
    patience=3,
    r=16,
    lora_alpha=32,
    lora_dropout=0.10,
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!



========== Tier 3: LoRA Fine-Tuning ==========
Backbone: xlm-roberta-base
LoRA rank: 16, alpha: 32
Total parameters: 278,643,664
Trainable parameters: 600,016
Trainable percentage: 0.2153%
Epoch 01 | train_loss=19.1661 | dev_f1=0.7387 | time=323.4s
  Saved best checkpoint.
Epoch 02 | train_loss=4.1562 | dev_f1=0.7793 | time=323.0s
  Saved best checkpoint.
Epoch 03 | train_loss=3.2252 | dev_f1=0.7973 | time=323.1s
  Saved best checkpoint.
Epoch 04 | train_loss=2.7754 | dev_f1=0.7897 | time=322.7s
Epoch 05 | train_loss=2.5025 | dev_f1=0.7955 | time=322.9s
Epoch 06 | train_loss=2.3994 | dev_f1=0.8015 | time=323.0s
  Saved best checkpoint.
Epoch 07 | train_loss=2.2660 | dev_f1=0.7999 | time=323.0s
Epoch 08 | train_loss=2.1423 | dev_f1=0.7880 | time=322.3s
Epoch 09 | train_loss=2.0562 | dev_f1=0.8091 | time=322.6s
